IDX CRMLS Pipeline — Clean Notebook Chunks
Paste each `

In [ ]:
` section into one notebook, or open this file directly in VS Code/Jupyter.

In [ ]:
# 1. Imports and path settings
from __future__ import annotations

import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path(r"C:\Users\User\Desktop\2026 Spring\IDX exchange DA")
RAW_DIR = BASE_DIR / "Raw Data"
OUTPUT_BASE = BASE_DIR / "Notebook Outputs"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

START_YYYYMM = "202401"
END_YYYYMM = f"{datetime.today().year}{datetime.today().month - 1:02d}" if datetime.today().month > 1 else f"{datetime.today().year - 1}12"
FILE_PATTERN = re.compile(r"^CRMLS(Listing|Sold)(\d{6})\.csv$", re.IGNORECASE)
DATE_COLS = ["CloseDate", "PurchaseContractDate", "ListingContractDate", "ContractStatusChangeDate"]
NUMERIC_COLS = [
    "ClosePrice", "ListPrice", "OriginalListPrice", "LivingArea", "BedroomsTotal",
    "BathroomsTotalInteger", "DaysOnMarket", "Latitude", "Longitude", "YearBuilt",
    "LotSizeAcres", "LotSizeSquareFeet",
]
CORE_FIELDS = {
    "transaction_price": ["ListPrice", "ClosePrice", "OriginalListPrice"],
    "property_size": ["LivingArea", "LotSizeSquareFeet", "LotSizeArea", "LotSizeAcres"],
    "property_config": ["BedroomsTotal", "BathroomsTotalInteger", "PropertyType", "PropertySubType"],
    "time_fields": ["CloseDate", "ListingContractDate", "PurchaseContractDate", "DaysOnMarket"],
    "location_fields": ["City", "PostalCode", "CountyOrParish", "Latitude", "Longitude", "UnparsedAddress"],
    "agent_brokerage": ["ListAgentFullName", "ListOfficeName", "BuyerOfficeName"],
    "listing_keys_ids": ["ListingKey", "ListingKeyNumeric", "ListingId"],
    "status_fields": ["MlsStatus", "ContractStatusChangeDate"],
}

In [ ]:
# 2. General helper functions

def read_csv_safe(path: Path) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            pass
    raise ValueError(f"Cannot read file: {path}")


def discover_files(data_dir: Path = RAW_DIR, start: str = START_YYYYMM, end: str = END_YYYYMM) -> pd.DataFrame:
    rows = []
    for path in data_dir.glob("*.csv"):
        m = FILE_PATTERN.match(path.name)
        if m and start <= m.group(2) <= end:
            rows.append({"file_name": path.name, "path": path, "dataset": m.group(1).capitalize(), "yyyymm": m.group(2)})
    return pd.DataFrame(rows).sort_values(["dataset", "yyyymm"]).reset_index(drop=True)


def add_missing_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def to_datetime_numeric(df: pd.DataFrame) -> pd.DataFrame:
    df = add_missing_cols(df, DATE_COLS + NUMERIC_COLS)
    for col in DATE_COLS:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def write_excel_sheets(path: Path, sheets: dict[str, pd.DataFrame]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)

In [ ]:
# 3. Combine monthly Listing/Sold files and keep Residential records

def combine_monthly_files(file_catalog: pd.DataFrame, dataset: str, residential_only: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    subset = file_catalog[file_catalog["dataset"].eq(dataset)].sort_values("yyyymm")
    frames, counts = [], []

    for row in subset.itertuples(index=False):
        df = read_csv_safe(Path(row.path))
        counts.append({"dataset": dataset, "file_name": row.file_name, "yyyymm": row.yyyymm, "rows": len(df), "cols": df.shape[1]})
        frames.append(df)

    if not frames:
        raise FileNotFoundError(f"No {dataset} files found.")

    combined = pd.concat(frames, ignore_index=True, sort=False)
    if residential_only and "PropertyType" in combined.columns:
        combined = combined[combined["PropertyType"].astype(str).str.strip().str.lower().eq("residential")].copy()

    return combined, pd.DataFrame(counts)


file_catalog = discover_files()
listing_raw, listing_counts = combine_monthly_files(file_catalog, "Listing", residential_only=True)
sold_raw, sold_counts = combine_monthly_files(file_catalog, "Sold", residential_only=True)

week1_dir = OUTPUT_BASE / "week1_combined"
week1_dir.mkdir(parents=True, exist_ok=True)
listing_raw.to_csv(week1_dir / "listing_residential_combined.csv", index=False, encoding="utf-8-sig")
sold_raw.to_csv(week1_dir / "sold_residential_combined.csv", index=False, encoding="utf-8-sig")
write_excel_sheets(week1_dir / "week1_file_log.xlsx", {
    "file_catalog": file_catalog,
    "listing_counts": listing_counts,
    "sold_counts": sold_counts,
})
print("Week 1 done:", len(listing_raw), "listing rows;", len(sold_raw), "sold rows")

In [ ]:
# 4. Schema profile, missingness, and data dictionary

def profile_columns(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        numeric_rate = pd.to_numeric(s, errors="coerce").notna().mean()
        date_rate = pd.to_datetime(s, errors="coerce").notna().mean() if "Date" in col else 0
        kind = "datetime" if date_rate > 0.7 else "numeric" if numeric_rate > 0.8 else "categorical" if s.nunique(dropna=True) <= 30 else "string"
        rows.append({
            "dataset": dataset,
            "column": col,
            "dtype": str(s.dtype),
            "missing_pct": s.isna().mean(),
            "nunique": s.nunique(dropna=True),
            "inferred_kind": kind,
            "sample_values": " | ".join(s.dropna().astype(str).head(5)),
        })
    return pd.DataFrame(rows)


def core_group(col: str) -> str:
    return next((g for g, cols in CORE_FIELDS.items() if col in cols), "other")


column_profile = pd.concat([
    profile_columns(listing_raw, "Listing"),
    profile_columns(sold_raw, "Sold"),
], ignore_index=True)

schema_compare = (
    column_profile.assign(present=1)
    .pivot_table(index="column", columns="dataset", values="present", aggfunc="max", fill_value=0)
    .reset_index()
)
schema_compare["comparison_flag"] = np.select(
    [schema_compare.get("Listing", 0).eq(1) & schema_compare.get("Sold", 0).eq(1), schema_compare.get("Listing", 0).eq(1)],
    ["both", "listing_only"],
    default="sold_only",
)

importance = column_profile.copy()
importance["core_group"] = importance["column"].map(core_group)
importance["completeness_score"] = (1 - importance["missing_pct"]) * 50
importance["type_score"] = np.where(importance["inferred_kind"].isin(["numeric", "datetime"]), 20, 10)
importance["business_score"] = np.where(importance["core_group"].ne("other"), 40, 0)
importance["importance_score"] = importance[["completeness_score", "type_score", "business_score"]].sum(axis=1)
importance["importance_tier"] = pd.cut(importance["importance_score"], [0, 50, 70, 90, np.inf], labels=["Tier 4", "Tier 3", "Tier 2", "Tier 1"])

week2_dir = OUTPUT_BASE / "week2_schema_profile"
write_excel_sheets(week2_dir / "schema_profile_outputs.xlsx", {
    "column_profile": column_profile,
    "schema_compare": schema_compare,
    "importance_rank": importance.sort_values(["dataset", "importance_score"], ascending=[True, False]),
})
print("Week 2 done:", week2_dir)

In [ ]:
# 5. Quick EDA charts: missingness, schema comparison, and data types

def save_bar(series: pd.Series, title: str, path: Path, ylabel: str = "Count") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ax = series.plot(kind="bar", figsize=(10, 5))
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


eda_dir = OUTPUT_BASE / "week2_eda_charts"
save_bar(column_profile.groupby("column")["missing_pct"].mean().sort_values(ascending=False).head(20),
         "Top 20 Columns by Missing Percentage", eda_dir / "missingness_top20.png", "Missing %")
save_bar(schema_compare["comparison_flag"].value_counts(), "Listing vs Sold Schema Comparison", eda_dir / "listing_vs_sold.png")
save_bar(column_profile.groupby(["dataset", "inferred_kind"]).size(), "Inferred Data Type Counts", eda_dir / "dtype_counts.png")
print("EDA charts saved:", eda_dir)

In [ ]:
# 6. Data cleaning and issue flags

ISSUE_INTERPRETATION = {
    "missing_close_date": "Listing may be unsold; for Sold data, review quality.",
    "missing_purchase_date": "Missing contract date.",
    "missing_listing_date": "Missing listing start date.",
    "missing_closeprice": "Missing transaction price.",
    "invalid_price": "ClosePrice <= 0.",
    "missing_livingarea": "Missing property size.",
    "invalid_area": "LivingArea <= 0.",
    "missing_dom": "Missing DaysOnMarket.",
    "invalid_dom": "DaysOnMarket < 0.",
    "listing_after_close": "Listing date after close date.",
    "purchase_after_close": "Purchase date after close date.",
    "negative_timeline": "Listing date after purchase contract date.",
    "missing_coord": "Missing latitude or longitude.",
    "zero_coord": "Latitude or longitude equals 0.",
    "positive_longitude": "Longitude positive; invalid for California.",
    "implausible_coord": "Coordinates outside broad California range.",
}


def clean_and_flag(df: pd.DataFrame, dataset: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = to_datetime_numeric(df)

    df["missing_closeprice"] = df["ClosePrice"].isna()
    df["invalid_price"] = df["ClosePrice"].notna() & df["ClosePrice"].le(0)
    df["missing_livingarea"] = df["LivingArea"].isna()
    df["invalid_area"] = df["LivingArea"].notna() & df["LivingArea"].le(0)
    df["missing_dom"] = df["DaysOnMarket"].isna()
    df["invalid_dom"] = df["DaysOnMarket"].notna() & df["DaysOnMarket"].lt(0)
    df["missing_listing_date"] = df["ListingContractDate"].isna()
    df["missing_purchase_date"] = df["PurchaseContractDate"].isna()
    df["missing_close_date"] = df["CloseDate"].isna()
    df["listing_after_close"] = df["ListingContractDate"].gt(df["CloseDate"])
    df["purchase_after_close"] = df["PurchaseContractDate"].gt(df["CloseDate"])
    df["negative_timeline"] = df["ListingContractDate"].gt(df["PurchaseContractDate"])
    df["missing_coord"] = df[["Latitude", "Longitude"]].isna().any(axis=1)
    df["zero_coord"] = df[["Latitude", "Longitude"]].eq(0).any(axis=1)
    df["positive_longitude"] = df["Longitude"].gt(0)
    df["implausible_coord"] = df["Latitude"].lt(30) | df["Latitude"].gt(43) | df["Longitude"].lt(-125) | df["Longitude"].gt(-113)

    issue_cols = list(ISSUE_INTERPRETATION)
    df[issue_cols] = df[issue_cols].fillna(False)
    df["dataset"] = dataset
    df["issue_count"] = df[issue_cols].sum(axis=1)
    df["issue_list"] = df[issue_cols].apply(lambda r: "; ".join(r.index[r]), axis=1)
    df["has_any_issue"] = df["issue_count"].gt(0)

    remove_flags = ["invalid_price", "invalid_area", "invalid_dom", "listing_after_close", "purchase_after_close", "negative_timeline", "zero_coord", "positive_longitude", "implausible_coord"]
    clean = df.loc[~df[remove_flags].any(axis=1)].copy()

    summary = pd.DataFrame({
        "dataset": dataset,
        "issue_flag": issue_cols,
        "count": [int(df[c].sum()) for c in issue_cols],
        "pct_of_rows": [df[c].mean() for c in issue_cols],
        "interpretation": [ISSUE_INTERPRETATION[c] for c in issue_cols],
    })
    size = pd.DataFrame([{"dataset": dataset, "before_rows": len(df), "after_rows": len(clean), "removed_rows": len(df) - len(clean)}])
    return df, clean, summary, size


listing_flagged, listing_clean, listing_issue_summary, listing_size = clean_and_flag(listing_raw, "Listing")
sold_flagged, sold_clean, sold_issue_summary, sold_size = clean_and_flag(sold_raw, "Sold")

week4_dir = OUTPUT_BASE / "week4_5_cleaning"
week4_dir.mkdir(parents=True, exist_ok=True)
listing_flagged.to_csv(week4_dir / "listing_full_flagged.csv", index=False, encoding="utf-8-sig")
sold_flagged.to_csv(week4_dir / "sold_full_flagged.csv", index=False, encoding="utf-8-sig")
listing_clean.to_csv(week4_dir / "listing_analysis_ready.csv", index=False, encoding="utf-8-sig")
sold_clean.to_csv(week4_dir / "sold_analysis_ready.csv", index=False, encoding="utf-8-sig")
write_excel_sheets(week4_dir / "cleaning_summary.xlsx", {
    "size_summary": pd.concat([listing_size, sold_size], ignore_index=True),
    "issue_summary": pd.concat([listing_issue_summary, sold_issue_summary], ignore_index=True),
})
print("Cleaning done:", week4_dir)

In [ ]:
# 7. Feature engineering and market metrics

def engineer_features(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    df = to_datetime_numeric(df)
    df["price_ratio"] = df["ClosePrice"] / df["OriginalListPrice"].where(df["OriginalListPrice"].gt(0))
    df["close_to_original_list_ratio"] = df["price_ratio"]
    df["price_per_sqft"] = df["ClosePrice"] / df["LivingArea"].where(df["LivingArea"].gt(0))
    df["year"] = df["CloseDate"].dt.year
    df["month"] = df["CloseDate"].dt.month
    df["yrmo"] = df["CloseDate"].dt.to_period("M").astype("string")
    df["listing_to_contract_days"] = (df["PurchaseContractDate"] - df["ListingContractDate"]).dt.days
    df["contract_to_close_days"] = (df["CloseDate"] - df["PurchaseContractDate"]).dt.days
    df["dataset"] = dataset
    return df


def summary_stats(df: pd.DataFrame, dataset: str, group_col: str | None = None) -> pd.DataFrame:
    metrics = ["ClosePrice", "price_per_sqft", "price_ratio", "DaysOnMarket", "listing_to_contract_days", "contract_to_close_days"]
    metrics = [c for c in metrics if c in df.columns]
    if group_col and group_col in df.columns:
        out = df.groupby(group_col)[metrics].agg(["count", "mean", "median"]).reset_index()
        out.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c for c in out.columns]
    else:
        out = df[metrics].describe().T.reset_index().rename(columns={"index": "metric"})
    out.insert(0, "dataset", dataset)
    return out


listing_fe = engineer_features(listing_clean, "Listing")
sold_fe = engineer_features(sold_clean, "Sold")
week6_dir = OUTPUT_BASE / "week6_features"
week6_dir.mkdir(parents=True, exist_ok=True)
listing_fe.to_csv(week6_dir / "listing_week6_feature_engineered.csv", index=False, encoding="utf-8-sig")
sold_fe.to_csv(week6_dir / "sold_week6_feature_engineered.csv", index=False, encoding="utf-8-sig")
write_excel_sheets(week6_dir / "week6_market_metrics.xlsx", {
    "overall": pd.concat([summary_stats(listing_fe, "Listing"), summary_stats(sold_fe, "Sold")], ignore_index=True),
    "monthly": pd.concat([summary_stats(listing_fe, "Listing", "yrmo"), summary_stats(sold_fe, "Sold", "yrmo")], ignore_index=True),
    "county_listing": summary_stats(listing_fe, "Listing", "CountyOrParish"),
    "county_sold": summary_stats(sold_fe, "Sold", "CountyOrParish"),
    "propertytype_listing": summary_stats(listing_fe, "Listing", "PropertyType"),
    "propertytype_sold": summary_stats(sold_fe, "Sold", "PropertyType"),
})
print("Feature engineering done:", week6_dir)

In [ ]:
# 8. Week 7 IQR outlier detection and final clean dataset

def add_iqr_flags(df: pd.DataFrame, dataset: str, cols: list[str] = ["ClosePrice", "LivingArea", "DaysOnMarket"]):
    df = df.copy()
    rows = []
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        flag = f"{col}_iqr_outlier_flag"
        df[flag] = df[col].notna() & (df[col].lt(lower) | df[col].gt(upper))
        rows.append({"dataset": dataset, "field": col, "q1": q1, "q3": q3, "iqr": iqr, "lower_bound": lower, "upper_bound": upper, "outlier_count": int(df[flag].sum()), "outlier_pct": df[flag].mean()})

    df["invalid_close_price_flag"] = df["ClosePrice"].notna() & df["ClosePrice"].le(0)
    df["invalid_living_area_flag"] = df["LivingArea"].notna() & df["LivingArea"].le(0)
    df["invalid_days_on_market_flag"] = df["DaysOnMarket"].notna() & df["DaysOnMarket"].lt(0)
    flag_cols = [c for c in df.columns if c.endswith("_flag")]
    df["week7_any_outlier_or_invalid_flag"] = df[flag_cols].any(axis=1)
    return df, pd.DataFrame(rows)


def before_after(original: pd.DataFrame, clean: pd.DataFrame, dataset: str) -> pd.DataFrame:
    rows = []
    for col in ["ClosePrice", "LivingArea", "DaysOnMarket"]:
        rows.append({
            "dataset": dataset,
            "metric": col,
            "before_row_count": len(original),
            "after_row_count": len(clean),
            "rows_removed": len(original) - len(clean),
            "rows_removed_pct": (len(original) - len(clean)) / len(original) if len(original) else 0,
            "before_median": original[col].median(skipna=True),
            "after_median": clean[col].median(skipna=True),
        })
    return pd.DataFrame(rows)


listing_w7, listing_iqr = add_iqr_flags(listing_fe, "Listing")
sold_w7, sold_iqr = add_iqr_flags(sold_fe, "Sold")
listing_final = listing_w7.loc[~listing_w7["week7_any_outlier_or_invalid_flag"]].copy()
sold_final = sold_w7.loc[~sold_w7["week7_any_outlier_or_invalid_flag"]].copy()

week7_dir = OUTPUT_BASE / "week7_outliers"
week7_dir.mkdir(parents=True, exist_ok=True)
listing_w7.to_csv(week7_dir / "listing_week7_full_flagged_dataset.csv", index=False, encoding="utf-8-sig")
sold_w7.to_csv(week7_dir / "sold_week7_full_flagged_dataset.csv", index=False, encoding="utf-8-sig")
listing_final.to_csv(week7_dir / "listing_week7_clean_filtered_dataset.csv", index=False, encoding="utf-8-sig")
sold_final.to_csv(week7_dir / "sold_week7_clean_filtered_dataset.csv", index=False, encoding="utf-8-sig")

flag_summary = pd.concat([
    pd.DataFrame({"dataset": "Listing", "flag": [c for c in listing_w7.columns if c.endswith("_flag")], "count": [int(listing_w7[c].sum()) for c in listing_w7.columns if c.endswith("_flag")]}),
    pd.DataFrame({"dataset": "Sold", "flag": [c for c in sold_w7.columns if c.endswith("_flag")], "count": [int(sold_w7[c].sum()) for c in sold_w7.columns if c.endswith("_flag")]}),
], ignore_index=True)

write_excel_sheets(week7_dir / "week7_outlier_detection_summary.xlsx", {
    "iqr_bounds": pd.concat([listing_iqr, sold_iqr], ignore_index=True),
    "flag_counts": flag_summary,
    "before_after": pd.concat([before_after(listing_w7, listing_final, "Listing"), before_after(sold_w7, sold_final, "Sold")], ignore_index=True),
    "listing_flagged_sample": listing_w7.head(100),
    "sold_flagged_sample": sold_w7.head(100),
})
print("Week 7 done:", week7_dir)
print("Final rows:", len(listing_final), "listing;", len(sold_final), "sold")